[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/notebooks/08-ai-engineering/02_multi-agent.ipynb)

*AI Engineering · Notebook 2 of 9*

# Agents & Multi-Agent Orchestration

How agent systems really work: the tool loop, routing, planning with dependencies, parallel fan-out, orchestrator–workers, handoffs, shared state, evaluator–optimizer loops, budgets, loop detection, worker pools and graph workflows like LangGraph.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** Agents are a team: a manager plans, specialists do the work, a reviewer checks it — and every extra person adds cost and coordination.

**What interviewers really test here:** Workflow vs agent, choosing the simplest pattern, handoffs, shared state, loop detection, budgets, and when multi-agent is worth it.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. What makes something an agent *(Basics)*
- [ ] 2. Patterns, from simple to autonomous *(Core)*
- [ ] 3. Planning, dependencies and parallelism *(Core)*
- [ ] 4. Handoffs, shared state and messages *(Core)*
- [ ] 5. Graph workflows (LangGraph and friends) *(Advanced)*
- [ ] 6. Cost, safety and when multi-agent is worth it *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline
import math
import json
import numpy as np
PLAIN = True

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### Workflow vs agent

|   | Workflow | Agent |
| --- | --- | --- |
| Who decides the steps | your code | the model |
| Predictability | high | lower |
| Cost and latency | bounded | variable |
| Best for | well-understood tasks | open-ended tasks |

### Failure modes to design for

- **Loops**: repeating the same tool call — detect repeats, cap steps.
- **Drift**: forgetting the goal over a long run — restate the goal, summarise progress.
- **Error cascades**: one agent's mistake becomes another's input — validate between steps.
- **Conflicting writes** to shared state — versions or single ownership.
- **Runaway cost** — token, step and time budgets.
- **Over-delegation** — orchestrators that spawn too many workers for simple tasks.

---
# Part 2: Lessons, step by step

## Lesson 1: What makes something an agent

*Level: Basics*

An **agent** is a model in a loop: it looks at the goal and what has happened so far, decides on an action (usually a tool call), sees the result, and repeats until it decides it's done. The model chooses the steps — that's the difference from a **workflow**, where your code fixes the steps in advance.



```python
import anthropic
client = anthropic.Anthropic()
messages = [{"role": "user", "content": goal}]
for step in range(MAX_STEPS):
    reply = client.messages.create(model="claude-opus-5-5", max_tokens=2000, tools=TOOLS, messages=messages)
    messages.append({"role": "assistant", "content": reply.content})
    if reply.stop_reason != "tool_use":
        break                                          # the model gave its final answer
    results = [{"type": "tool_result", "tool_use_id": b.id, "content": run_tool(b.name, b.input)}
               for b in reply.content if b.type == "tool_use"]
    messages.append({"role": "user", "content": results})
```



Always cap the number of steps: a confused agent will otherwise loop, burning tokens. The Claude SDKs also ship a tool runner and the Claude Agent SDK, which run this loop for you.

**✏️ Your turn:** you now have what you need for [Q1](#q1), [Q6](#q6). Try one now, or carry on and do them in Part 3.

## Lesson 2: Patterns, from simple to autonomous

*Level: Core*

| Pattern | How it works | Use when |
| --- | --- | --- |
| **Prompt chain** | fixed steps, each call uses the last one's output | the steps are known (outline → draft → translate) |
| **Router** | classify, then send to a specialised prompt or model | distinct kinds of request (billing vs tech support) |
| **Parallel fan-out** | independent sub-tasks at once, then combine (or vote) | parts don't depend on each other; you want speed or several opinions |
| **Orchestrator–workers** | a lead model plans sub-tasks at run time and delegates | you can't know the sub-tasks in advance (research, big code changes) |
| **Evaluator–optimizer** | one model drafts, another critiques, repeat | clear quality criteria and iteration helps |
| **Autonomous agent** | model loops with tools until done | open-ended tasks with an unknown number of steps |



The golden rule: **use the simplest pattern that solves the problem**. Each step up adds cost, latency and failure modes. Many "agents" in production are a router plus two prompt chains.

**✏️ Your turn:** you now have what you need for [Q11](#q11), [Q2](#q2), [Q3](#q3), [Q4](#q4). Try one now, or carry on and do them in Part 3.

## Lesson 3: Planning, dependencies and parallelism

*Level: Core*

An orchestrator first **plans**: it breaks the goal into sub-tasks and notes which depend on which. That's a graph, so the run order is a **topological sort**, and independent tasks can run at the same time. The time to finish is the **critical path** (the longest chain of dependent tasks), not the sum of all tasks.



```python
import asyncio
async def run_level(tasks):                       # everything whose dependencies are done
    return await asyncio.gather(*(worker(t) for t in tasks))
```



Real systems also cap concurrency (API rate limits), so workers form a **pool**, and they set **timeouts** so one slow worker doesn't hold up the answer — returning partial results is usually better than waiting.

**✏️ Your turn:** you now have what you need for [Q12](#q12), [Q13](#q13), [Q19](#q19), [Q8](#q8), [Q18](#q18). Try one now, or carry on and do them in Part 3.

## Lesson 4: Handoffs, shared state and messages

*Level: Core*

Agents coordinate in three main ways:




- **Handoffs**: the current agent transfers the conversation to a specialist (triage → billing → refunds). Guard against ping-pong loops and endless chains, and pass a **summary plus recent messages**, not the whole history.
- **Shared state** (a "blackboard"): agents read and write a common store. Concurrent writers can overwrite each other, so use versions (optimistic locking) or a single owner per key.
- **Messages**: agents send each other messages through inboxes — flexible but hard to debug, so log every message.

**✏️ Your turn:** you now have what you need for [Q14](#q14), [Q9](#q9), [Q15](#q15), [Q16](#q16), [Q10](#q10). Try one now, or carry on and do them in Part 3.

## Lesson 5: Graph workflows (LangGraph and friends)

*Level: Advanced*

Frameworks such as LangGraph model an agent system as a **state machine**: nodes are steps (often model calls), edges say what comes next, and **conditional edges** decide based on the state. This makes loops, retries and human approval steps explicit, and the state can be saved (checkpointed) so a run can pause and resume.



```python
from langgraph.graph import StateGraph, START, END
g = StateGraph(State)
g.add_node("draft", draft); g.add_node("review", review)
g.add_edge(START, "draft"); g.add_edge("draft", "review")
g.add_conditional_edges("review", lambda s: END if s["score"] >= 8 else "draft")
app = g.compile(); app.invoke({"topic": "MCP"})
```



Whatever the framework, the ideas are the ones you build by hand in this topic — knowing them lets you debug any framework, or skip it.

**✏️ Your turn:** you now have what you need for [Q20](#q20). Try one now, or carry on and do them in Part 3.

## Lesson 6: Cost, safety and when multi-agent is worth it

*Level: Advanced*

- **Cost multiplies**: each agent re-reads context. A four-worker system can easily use five times the tokens of one agent. It pays off when sub-tasks are truly independent and parallelism or specialisation improves quality.
- **Budgets**: cap tokens, steps and wall-clock time per run, and per agent.
- **Least privilege**: each agent gets only the tools it needs; a researcher doesn't need write access.
- **Observability**: log every model call, tool call and handoff with a shared trace id (see the Observability topic) — multi-agent bugs are invisible otherwise.
- **Evaluate the whole system** on real tasks, not each agent in isolation.

**✏️ Your turn:** you now have what you need for [Q17](#q17), [Q5](#q5), [Q7](#q7). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

**▶ Run this cell before the questions.** It sets up the data and tools they share.

In [2]:
from collections import Counter, deque

import heapq


<a id="q1"></a>
### Q1. Run the Agent Loop

🟢 Warm-up · *Tool use until done*

A scripted model returns, at each step, either `('tool', name, arg)` or `('final', text)`. Run the loop: call tools from `TOOLS`, stop on 'final' or after `max_steps` model calls. Return `(final text or 'stopped: max steps', tool results in order)`.

**Examples**

- `script = [('tool', 'search', 'mcp'), ('tool', 'calc', '2*21'), ('final', 'done')], max_steps = 5` → `('done', ['3 results for mcp', '42'])`
- `script = [('tool', 'search', 'x'), ('tool', 'search', 'x'), ('tool', 'search', 'x'), ('tool', 'search', 'x')], max_steps = 3` → `('stopped: max steps', ['3 results for x', '3 results for x', '3 results for x'])`

In [ ]:
TOOLS = {"search": lambda q: f"3 results for {q}", "calc": lambda e: str(eval(e, {"__builtins__": {}}))}

def agent_loop(script, max_steps):
    """
    A scripted model returns, at each step, either ('tool', name, arg) or
    ('final', text). Run the loop: call tools from TOOLS, stop on 'final' or
    after max_steps model calls. Return (final text or 'stopped: max steps',
    tool results in order).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(agent_loop, [
    (([('tool', 'search', 'mcp'), ('tool', 'calc', '2*21'), ('final', 'done')], 5), ('done', ['3 results for mcp', '42'])),
    (([('tool', 'search', 'x'), ('tool', 'search', 'x'), ('tool', 'search', 'x'), ('tool', 'search', 'x')], 3), ('stopped: max steps', ['3 results for x', '3 results for x', '3 results for x'])),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A while loop with a step counter.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A for loop over `range(max_steps)` with an early return.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** A while loop with a step counter.

```python
def agent_loop(script, max_steps):
    results = []
    step = 0
    while step < max_steps:
        action = script[step]
        step += 1
        if action[0] == "final":
            return (action[1], results)
        results.append(TOOLS[action[1]](action[2]))
    return ("stopped: max steps", results)
```

**Cost:** O(steps) time, O(steps) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A for loop over `range(max_steps)` with an early return.

```python
def agent_loop(script, max_steps):
    results = []
    for action in script[:max_steps]:
        if action[0] == "final":
            return (action[1], results)
        results.append(TOOLS[action[1]](action[2]))
    return ("stopped: max steps", results)
```

**Cost:** O(steps) time, O(steps) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A while loop with a step counter. That costs O(steps) time and O(steps) space."

"I can do better: A for loop over `range(max_steps)` with an early return. That's O(steps) time and O(steps) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/multi-agent/run-the-agent-loop)

<a id="q2"></a>
### Q2. Route to a Specialist

🟢 Warm-up · *Router pattern*

Send a request to the specialist whose keywords appear most often in it (words are lower-cased, punctuation stripped). Ties go to the earlier route in `ROUTES`; no match goes to 'general'.

**Examples**

- `text = 'I was charged twice, please refund the payment'` → `'billing'`
- `text = 'App crash after login'` → `'tech'`
- `text = 'Hello there'` → `'general'`

In [ ]:
ROUTES = {"billing": ["refund", "invoice", "charge", "payment"], "tech": ["error", "crash", "login", "bug"],
          "sales": ["price", "plan", "upgrade", "discount"]}

def route(text):
    """
    Send a request to the specialist whose keywords appear most often in it
    (words are lower-cased, punctuation stripped). Ties go to the earlier route
    in ROUTES; no match goes to 'general'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(route, [
    (('I was charged twice, please refund the payment',), 'billing'),
    (('App crash after login',), 'tech'),
    (('Hello there',), 'general'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count matches per route with nested loops.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A Counter of words, then sum per route and `max`.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Count matches per route with nested loops.

```python
def route(text):
    words = [w.strip(".,!?").lower() for w in text.split()]
    best, best_n = "general", 0
    for name, keys in ROUTES.items():
        n = 0
        for w in words:
            if w in keys:
                n += 1
        if n > best_n:
            best, best_n = name, n
    return best
```

**Cost:** O(w · k) time, O(r) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A Counter of words, then sum per route and `max`.

```python
def route(text):
    counts = Counter(w.strip(".,!?").lower() for w in text.split())
    scores = {name: sum(counts[k] for k in keys) for name, keys in ROUTES.items()}
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else "general"
```

**Cost:** O(w + k) time, O(w) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count matches per route with nested loops. That costs O(w · k) time and O(r) space."

"I can do better: A Counter of words, then sum per route and `max`. That's O(w + k) time and O(w) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/multi-agent/route-to-a-specialist)

<a id="q3"></a>
### Q3. Combine Worker Answers by Vote

🟢 Warm-up · *Orchestrator–workers*

Several workers answered the same question. Return `(winning answer, share of votes rounded to 2)`; ties go to the answer seen first. Answers are compared after stripping and lower-casing.

**Examples**

- `answers = ['Paris', 'paris ', 'Lyon', 'Paris']` → `('paris', 0.75)`
- `answers = ['A', 'B']` → `('a', 0.5)`

In [ ]:
def vote(answers):
    """
    Several workers answered the same question. Return (winning answer, share of
    votes rounded to 2); ties go to the answer seen first. Answers are compared
    after stripping and lower-casing.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(vote, [
    ((['Paris', 'paris ', 'Lyon', 'Paris'],), ('paris', 0.75)),
    ((['A', 'B'],), ('a', 0.5)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count in a dict and track the first-seen order.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`Counter.most_common` keeps first-seen order for ties.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Count in a dict and track the first-seen order.

```python
def vote(answers):
    counts, order = {}, []
    for a in answers:
        a = a.strip().lower()
        if a not in counts:
            counts[a] = 0
            order.append(a)
        counts[a] += 1
    best = order[0]
    for a in order:
        if counts[a] > counts[best]:
            best = a
    return (best, round(counts[best] / len(answers), 2))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `Counter.most_common` keeps first-seen order for ties.

```python
def vote(answers):
    best, n = Counter(a.strip().lower() for a in answers).most_common(1)[0]
    return (best, round(n / len(answers), 2))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count in a dict and track the first-seen order. That costs O(n) time and O(n) space."

"I can do better: `Counter.most_common` keeps first-seen order for ties. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/multi-agent/combine-worker-answers-by-vote)

<a id="q4"></a>
### Q4. Evaluator–Optimizer Loop

🟢 Warm-up · *Generate, critique, retry*

A generator produces drafts; an evaluator scores each (here `scores[i]` for draft i). Stop at the first draft scoring ≥ `target`, or after `max_rounds` drafts and return the best one seen. Return `(draft index, score, rounds used)`.

**Examples**

- `scores = [0.4, 0.7, 0.9, 0.95], target = 0.85, max_rounds = 5` → `(2, 0.9, 3)`
- `scores = [0.4, 0.7, 0.6], target = 0.85, max_rounds = 3` → `(1, 0.7, 3)`

In [ ]:
def refine(scores, target, max_rounds):
    """
    A generator produces drafts; an evaluator scores each (here scores[i] for
    draft i). Stop at the first draft scoring ≥ target, or after max_rounds
    drafts and return the best one seen. Return (draft index, score, rounds
    used).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(refine, [
    (([0.4, 0.7, 0.9, 0.95], 0.85, 5), (2, 0.9, 3)),
    (([0.4, 0.7, 0.6], 0.85, 3), (1, 0.7, 3)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A loop tracking the best so far.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Find the first passing index with `next`; otherwise `max` over the tried drafts.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** A loop tracking the best so far.

```python
def refine(scores, target, max_rounds):
    best = 0
    for i in range(min(max_rounds, len(scores))):
        if scores[i] > scores[best]:
            best = i
        if scores[i] >= target:
            return (i, scores[i], i + 1)
    return (best, scores[best], min(max_rounds, len(scores)))
```

**Cost:** O(r) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Find the first passing index with `next`; otherwise `max` over the tried drafts.

```python
def refine(scores, target, max_rounds):
    tried = scores[:max_rounds]
    hit = next((i for i, s in enumerate(tried) if s >= target), None)
    if hit is not None:
        return (hit, tried[hit], hit + 1)
    best = max(range(len(tried)), key=tried.__getitem__)
    return (best, tried[best], len(tried))
```

**Cost:** O(r) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A loop tracking the best so far. That costs O(r) time and O(1) space."

"I can do better: Find the first passing index with `next`; otherwise `max` over the tried drafts. That's O(r) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/multi-agent/evaluator-optimizer-loop)

<a id="q5"></a>
### Q5. Stay Within the Token Budget

🟢 Warm-up · *Cost control*

Run tasks in order; each costs `tokens[task]`. Skip a task that would push the total past `budget`, but keep trying smaller later tasks. Return `(done tasks, tokens used)`.

**Examples**

- `tokens = {'plan': 2000, 'research': 9000, 'draft': 4000, 'polish': 1500}, budget = 8000` → `(['plan', 'draft', 'polish'], 7500)`

In [ ]:
def within_budget(tokens, budget):
    """
    Run tasks in order; each costs tokens[task]. Skip a task that would push the
    total past budget, but keep trying smaller later tasks. Return (done tasks,
    tokens used).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(within_budget, [
    (({'plan': 2000, 'research': 9000, 'draft': 4000, 'polish': 1500}, 8000), (['plan', 'draft', 'polish'], 7500)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A running total in a loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same loop; a list comprehension can't carry the running total cleanly, so a loop is the right tool.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** A running total in a loop.

```python
def within_budget(tokens, budget):
    done, used = [], 0
    for task, cost in tokens.items():
        if used + cost <= budget:
            done.append(task)
            used += cost
    return (done, used)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Same loop; a list comprehension can't carry the running total cleanly, so a loop is the right tool.

```python
def within_budget(tokens, budget):
    done, used = [], 0
    for task, cost in tokens.items():
        if used + cost > budget:
            continue
        done.append(task)
        used += cost
    return (done, used)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A running total in a loop. That costs O(n) time and O(n) space."

"I can do better: Same loop; a list comprehension can't carry the running total cleanly, so a loop is the right tool. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/multi-agent/stay-within-the-token-budget)

<a id="q6"></a>
### Q6. Catch an Agent Going in Circles

🟢 Warm-up · *Loop detection*

Return the index of the first step whose `(tool, argument)` already happened earlier — a sign the agent is stuck — or -1.

**Examples**

- `steps = [('search', 'mcp'), ('read', 'a.md'), ('search', 'mcp spec'), ('read', 'a.md')]` → `3`
- `steps = [('a', 1), ('b', 2)]` → `-1`

In [ ]:
def first_repeat(steps):
    """
    Return the index of the first step whose (tool, argument) already happened
    earlier — a sign the agent is stuck — or -1.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(first_repeat, [
    (([('search', 'mcp'), ('read', 'a.md'), ('search', 'mcp spec'), ('read', 'a.md')],), 3),
    (([('a', 1), ('b', 2)],), -1),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compare with every earlier step.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A set of seen steps.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Compare with every earlier step.

```python
def first_repeat(steps):
    for i in range(len(steps)):
        for j in range(i):
            if steps[j] == steps[i]:
                return i
    return -1
```

**Cost:** O(n²) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A set of seen steps.

```python
def first_repeat(steps):
    seen = set()
    for i, s in enumerate(steps):
        if s in seen:
            return i
        seen.add(s)
    return -1
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compare with every earlier step. That costs O(n²) time and O(1) space."

"I can do better: A set of seen steps. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/multi-agent/catch-an-agent-going-in-circles)

<a id="q7"></a>
### Q7. Each Agent Only Gets Its Own Tools

🟢 Warm-up · *Least privilege*

`allowed[agent]` lists the tools an agent may use. Return the calls `(agent, tool)` that must be refused, in order.

**Examples**

- `allowed = {'researcher': ['search', 'read'], 'writer': ['write_file']}, calls = [('researcher', 'search'), ('writer', 'search'), ('researcher', 'write_file'), ('writer', 'write_file')]` → `[('writer', 'search'), ('researcher', 'write_file')]`

In [ ]:
def refused(allowed, calls):
    """
    allowed[agent] lists the tools an agent may use. Return the calls (agent,
    tool) that must be refused, in order.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(refused, [
    (({'researcher': ['search', 'read'], 'writer': ['write_file']}, [('researcher', 'search'), ('writer', 'search'), ('researcher', 'write_file'), ('writer', 'write_file')]), [('writer', 'search'), ('researcher', 'write_file')]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A loop with an if.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Convert to sets once; a comprehension filters.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** A loop with an if.

```python
def refused(allowed, calls):
    out = []
    for agent, tool in calls:
        if tool not in allowed.get(agent, []):
            out.append((agent, tool))
    return out
```

**Cost:** O(c) time, O(c) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Convert to sets once; a comprehension filters.

```python
def refused(allowed, calls):
    sets = {a: set(t) for a, t in allowed.items()}
    return [(a, t) for a, t in calls if t not in sets.get(a, set())]
```

**Cost:** O(c) time, O(a + c) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A loop with an if. That costs O(c) time and O(c) space."

"I can do better: Convert to sets once; a comprehension filters. That's O(c) time and O(a + c) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/multi-agent/each-agent-only-gets-its-own-tools)

<a id="q8"></a>
### Q8. Workers That Time Out

🟢 Warm-up · *Partial results*

Fan out to workers that take `secs` seconds each; wait at most `timeout`. Return `(finished workers, timed-out workers)`, each list sorted. A good orchestrator answers with partial results instead of waiting forever.

**Examples**

- `secs = {'news': 2.5, 'sales': 7.0, 'weather': 1.2, 'stocks': 5.0}, timeout = 5.0` → `(['news', 'stocks', 'weather'], ['sales'])`

In [ ]:
def with_timeout(secs, timeout):
    """
    Fan out to workers that take secs seconds each; wait at most timeout. Return
    (finished workers, timed-out workers), each list sorted. A good orchestrator
    answers with partial results instead of waiting forever.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(with_timeout, [
    (({'news': 2.5, 'sales': 7.0, 'weather': 1.2, 'stocks': 5.0}, 5.0), (['news', 'stocks', 'weather'], ['sales'])),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Two lists filled in a loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Two sorted comprehensions.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Two lists filled in a loop.

```python
def with_timeout(secs, timeout):
    done, late = [], []
    for w, s in secs.items():
        if s <= timeout:
            done.append(w)
        else:
            late.append(w)
    return (sorted(done), sorted(late))
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Two sorted comprehensions.

```python
def with_timeout(secs, timeout):
    return (sorted(w for w, s in secs.items() if s <= timeout), sorted(w for w, s in secs.items() if s > timeout))
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Two lists filled in a loop. That costs O(n log n) time and O(n) space."

"I can do better: Two sorted comprehensions. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/multi-agent/workers-that-time-out)

<a id="q9"></a>
### Q9. Hand Off Only What's Needed

🟢 Warm-up · *Context on handoff*

When agent A hands off to agent B, send a summary (`summary_tokens`) plus only the last `k` messages, instead of the whole history. Return `(tokens if sending everything, tokens sent, % saved rounded to 1)`.

**Examples**

- `messages = [1200, 800, 3000, 450, 600, 900], k = 2, summary_tokens = 300` → `(6950, 1800, 74.1)`

In [ ]:
def handoff_tokens(messages, k, summary_tokens):
    """
    When agent A hands off to agent B, send a summary (summary_tokens) plus only
    the last k messages, instead of the whole history. Return (tokens if sending
    everything, tokens sent, % saved rounded to 1).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(handoff_tokens, [
    (([1200, 800, 3000, 450, 600, 900], 2, 300), (6950, 1800, 74.1)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Add up in loops.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`sum` with slicing.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Add up in loops.

```python
def handoff_tokens(messages, k, summary_tokens):
    full = 0
    for m in messages:
        full += m
    sent = summary_tokens
    for m in messages[len(messages) - k:]:
        sent += m
    return (full, sent, round((full - sent) / full * 100, 1))
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `sum` with slicing.

```python
def handoff_tokens(messages, k, summary_tokens):
    full, sent = sum(messages), summary_tokens + sum(messages[-k:])
    return (full, sent, round((full - sent) / full * 100, 1))
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Add up in loops. That costs O(n) time and O(1) space."

"I can do better: `sum` with slicing. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/multi-agent/hand-off-only-whats-needed)

<a id="q10"></a>
### Q10. Merge Sources From Workers

🟢 Warm-up · *Citations*

Each worker returns the sources it used. Return one de-duplicated list in order of first appearance.

**Examples**

- `lists = [['kb/1', 'kb/4'], ['kb/4', 'web/2'], [], ['kb/1', 'kb/9']]` → `['kb/1', 'kb/4', 'web/2', 'kb/9']`

In [ ]:
def merge_sources(lists):
    """
    Each worker returns the sources it used. Return one de-duplicated list in
    order of first appearance.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(merge_sources, [
    (([['kb/1', 'kb/4'], ['kb/4', 'web/2'], [], ['kb/1', 'kb/9']],), ['kb/1', 'kb/4', 'web/2', 'kb/9']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A list plus a membership check.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`dict.fromkeys` keeps first-seen order and removes duplicates.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** A list plus a membership check.

```python
def merge_sources(lists):
    out = []
    for lst in lists:
        for s in lst:
            if s not in out:
                out.append(s)
    return out
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `dict.fromkeys` keeps first-seen order and removes duplicates.

```python
def merge_sources(lists):
    return list(dict.fromkeys(s for lst in lists for s in lst))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A list plus a membership check. That costs O(n²) time and O(n) space."

"I can do better: `dict.fromkeys` keeps first-seen order and removes duplicates. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/multi-agent/merge-sources-from-workers)

<a id="q11"></a>
### Q11. Pick the Simplest Pattern That Works

🟢 Warm-up · *Design judgement*

Choose a pattern from a task's traits, checking in this order: if `steps_known` and not `needs_tools` → 'single call'; if `steps_known` → 'prompt chain'; if `categories` → 'router'; if `independent_parts` → 'parallel fan-out'; if `needs_quality_loop` → 'evaluator-optimizer'; otherwise 'autonomous agent'.

**Examples**

- `traits = {'steps_known': True, 'needs_tools': False}` → `'single call'`
- `traits = {'categories': True}` → `'router'`
- `traits = {'independent_parts': True, 'needs_quality_loop': True}` → `'parallel fan-out'`

In [ ]:
def choose_pattern(traits):
    """
    Choose a pattern from a task's traits, checking in this order: if
    steps_known and not needs_tools → 'single call'; if steps_known → 'prompt
    chain'; if categories → 'router'; if independent_parts → 'parallel fan-out';
    if needs_quality_loop → 'evaluator-optimizer'; otherwise 'autonomous agent'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(choose_pattern, [
    (({'steps_known': True, 'needs_tools': False},), 'single call'),
    (({'categories': True},), 'router'),
    (({'independent_parts': True, 'needs_quality_loop': True},), 'parallel fan-out'),
    (({},), 'autonomous agent'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

An if-chain with `.get(key, False)`.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

An ordered list of (condition, pattern) rules and `next`.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** An if-chain with `.get(key, False)`.

```python
def choose_pattern(traits):
    if traits.get("steps_known", False) and not traits.get("needs_tools", False):
        return "single call"
    if traits.get("steps_known", False):
        return "prompt chain"
    if traits.get("categories", False):
        return "router"
    if traits.get("independent_parts", False):
        return "parallel fan-out"
    if traits.get("needs_quality_loop", False):
        return "evaluator-optimizer"
    return "autonomous agent"
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** An ordered list of (condition, pattern) rules and `next`.

```python
def choose_pattern(traits):
    t = lambda k: traits.get(k, False)
    rules = [(t("steps_known") and not t("needs_tools"), "single call"), (t("steps_known"), "prompt chain"),
             (t("categories"), "router"), (t("independent_parts"), "parallel fan-out"),
             (t("needs_quality_loop"), "evaluator-optimizer")]
    return next((name for ok, name in rules if ok), "autonomous agent")
```

**Cost:** O(r) time, O(r) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: An if-chain with `.get(key, False)`. That costs O(1) time and O(1) space."

"I can do better: An ordered list of (condition, pattern) rules and `next`. That's O(r) time and O(r) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/multi-agent/pick-the-simplest-pattern-that-works)

<a id="q12"></a>
### Q12. Order a Plan

🟡 Interview standard · *Dependencies · topological sort*

A planner produced tasks with dependencies (task → tasks it needs first). Return an order that respects them; among tasks that are ready at the same time, take them alphabetically.

**Examples**

- `deps = {'write_report': ['analyse', 'fetch_news'], 'analyse': ['fetch_sales', 'fetch_news'], 'fetch_sales': [], 'fetch_news': []}` → `['fetch_news', 'fetch_sales', 'analyse', 'write_report']`
- `deps = {'c': ['a'], 'b': ['a'], 'a': []}` → `['a', 'b', 'c']`

In [ ]:
from graphlib import TopologicalSorter

def plan_order(deps):
    """
    A planner produced tasks with dependencies (task → tasks it needs first).
    Return an order that respects them; among tasks that are ready at the same
    time, take them alphabetically.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(plan_order, [
    (({'write_report': ['analyse', 'fetch_news'], 'analyse': ['fetch_sales', 'fetch_news'], 'fetch_sales': [], 'fetch_news': []},), ['fetch_news', 'fetch_sales', 'analyse', 'write_report']),
    (({'c': ['a'], 'b': ['a'], 'a': []},), ['a', 'b', 'c']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Kahn's algorithm: repeatedly take the alphabetically-first task whose needs are done.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`graphlib.TopologicalSorter` plus a min-heap, so the alphabetically-first ready task always goes next.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Kahn's algorithm: repeatedly take the alphabetically-first task whose needs are done.

```python
def plan_order(deps):
    done, order = set(), []
    while len(order) < len(deps):
        ready = sorted(t for t in deps if t not in done and all(d in done for d in deps[t]))
        order.append(ready[0])
        done.add(ready[0])
    return order
```

**Cost:** O(n² ) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `graphlib.TopologicalSorter` plus a min-heap, so the alphabetically-first ready task always goes next.

```python
def plan_order(deps):
    ts = TopologicalSorter(deps)
    ts.prepare()
    ready, order = [], []
    while ts.is_active():
        for t in ts.get_ready():      # each task is offered once, so keep it in a heap
            heapq.heappush(ready, t)
        t = heapq.heappop(ready)
        order.append(t)
        ts.done(t)
    return order
```

**Cost:** O(n log n + e) time, O(n + e) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Kahn's algorithm: repeatedly take the alphabetically-first task whose needs are done. That costs O(n² ) time and O(n) space."

"I can do better: `graphlib.TopologicalSorter` plus a min-heap, so the alphabetically-first ready task always goes next. That's O(n log n + e) time and O(n + e) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/multi-agent/order-a-plan)

<a id="q13"></a>
### Q13. How Long With Parallel Workers?

🟡 Interview standard · *Fan-out · critical path*

Each task takes `secs[task]` and can start as soon as its dependencies finish (unlimited workers). Return `(total time if run one by one, total time in parallel)`.

**Examples**

- `deps = {'write_report': ['analyse', 'fetch_news'], 'analyse': ['fetch_sales', 'fetch_news'], 'fetch_sales': [], 'fetch_news': []}, secs = {'fetch_sales': 4, 'fetch_news': 6, 'analyse': 3, 'write_report': 2}` → `(15, 11)`

In [ ]:
def wall_time(deps, secs):
    """
    Each task takes secs[task] and can start as soon as its dependencies finish
    (unlimited workers). Return (total time if run one by one, total time in
    parallel).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(wall_time, [
    (({'write_report': ['analyse', 'fetch_news'], 'analyse': ['fetch_sales', 'fetch_news'], 'fetch_sales': [], 'fetch_news': []}, {'fetch_sales': 4, 'fetch_news': 6, 'analyse': 3, 'write_report': 2}), (15, 11)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Recursively compute each task's finish time = its time + the latest dependency's finish.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same recurrence with `functools.cache`, written as one expression.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Recursively compute each task's finish time = its time + the latest dependency's finish.

```python
def wall_time(deps, secs):
    finish = {}

    def end(t):
        if t not in finish:
            latest = 0
            for d in deps[t]:
                latest = max(latest, end(d))
            finish[t] = latest + secs[t]
        return finish[t]
    return (sum(secs.values()), max(end(t) for t in deps))
```

**Cost:** O(n + e) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Same recurrence with `functools.cache`, written as one expression.

```python
def wall_time(deps, secs):
    from functools import cache

    @cache
    def end(t):
        return secs[t] + max((end(d) for d in deps[t]), default=0)
    return (sum(secs.values()), max(map(end, deps)))
```

**Cost:** O(n + e) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Recursively compute each task's finish time = its time + the latest dependency's finish. That costs O(n + e) time and O(n) space."

"I can do better: Same recurrence with `functools.cache`, written as one expression. That's O(n + e) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `deps = {'write_report': ['analyse', 'fetch_news'], 'analyse': ['fetch_sales', 'fetch_news'], 'fetch_sales': [], 'fetch_news': []}, secs = {'fetch_sales': 4, 'fetch_news': 6, 'analyse': 3, 'write_report': 2}`

1. end() returns 4
2. end() returns 6
3. end() returns 9
4. end() returns 11
5. wall\_time() returns (15, 11)

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/multi-agent/how-long-with-parallel-workers)

<a id="q14"></a>
### Q14. Follow the Handoffs

🟡 Interview standard · *Handoff pattern*

Agents hand the conversation to each other. `handoffs[agent]` is the agent it passes to, or None if it answers itself. Start at `start`; return the path of agents, or the path ending in 'LOOP' if an agent repeats, or 'TOO DEEP' after `max_hops` handoffs.

**Examples**

- `handoffs = {'triage': 'billing', 'billing': 'refunds', 'refunds': None}, start = 'triage', max_hops = 5` → `['triage', 'billing', 'refunds']`
- `handoffs = {'a': 'b', 'b': 'a'}, start = 'a', max_hops = 5` → `['a', 'b', 'LOOP']`
- `handoffs = {'a': 'b', 'b': 'c', 'c': 'd', 'd': None}, start = 'a', max_hops = 2` → `['a', 'b', 'c', 'TOO DEEP']`

In [ ]:
def follow(handoffs, start, max_hops):
    """
    Agents hand the conversation to each other. handoffs[agent] is the agent it
    passes to, or None if it answers itself. Start at start; return the path of
    agents, or the path ending in 'LOOP' if an agent repeats, or 'TOO DEEP'
    after max_hops handoffs.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(follow, [
    (({'triage': 'billing', 'billing': 'refunds', 'refunds': None}, 'triage', 5), ['triage', 'billing', 'refunds']),
    (({'a': 'b', 'b': 'a'}, 'a', 5), ['a', 'b', 'LOOP']),
    (({'a': 'b', 'b': 'c', 'c': 'd', 'd': None}, 'a', 2), ['a', 'b', 'c', 'TOO DEEP']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A loop with a visited list.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same loop with a set for O(1) repeat checks.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** A loop with a visited list.

```python
def follow(handoffs, start, max_hops):
    path = [start]
    hops = 0
    while handoffs[path[-1]] is not None:
        nxt = handoffs[path[-1]]
        if nxt in path:
            return path + ["LOOP"]
        hops += 1
        if hops > max_hops:
            return path + ["TOO DEEP"]
        path.append(nxt)
    return path
```

**Cost:** O(h) time, O(h) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Same loop with a set for O(1) repeat checks.

```python
def follow(handoffs, start, max_hops):
    path, seen = [start], {start}
    while (nxt := handoffs[path[-1]]) is not None:
        if nxt in seen:
            return path + ["LOOP"]
        if len(path) > max_hops:
            return path + ["TOO DEEP"]
        path.append(nxt)
        seen.add(nxt)
    return path
```

**Cost:** O(h) time, O(h) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A loop with a visited list. That costs O(h) time and O(h) space."

"I can do better: Same loop with a set for O(1) repeat checks. That's O(h) time and O(h) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/multi-agent/follow-the-handoffs)

<a id="q15"></a>
### Q15. Shared State Without Lost Updates

🟡 Interview standard · *Blackboard · optimistic locking*

Agents write to a shared state. Each update is `(agent, key, value, version_seen)`; it applies only if `version_seen` equals the key's current version (starting at 0), and then the version goes up by 1. Return `(final state, rejected agents in order)`.

**Examples**

- `updates = [('planner', 'plan', 'v1', 0), ('coder', 'plan', 'v2', 0), ('coder', 'code', 'x=1', 0), ('reviewer', 'plan', 'v3', 1)]` → `({'plan': 'v3', 'code': 'x=1'}, ['coder'])`

In [ ]:
def apply_updates(updates):
    """
    Agents write to a shared state. Each update is (agent, key, value,
    version_seen); it applies only if version_seen equals the key's current
    version (starting at 0), and then the version goes up by 1. Return (final
    state, rejected agents in order).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(apply_updates, [
    (([('planner', 'plan', 'v1', 0), ('coder', 'plan', 'v2', 0), ('coder', 'code', 'x=1', 0), ('reviewer', 'plan', 'v3', 1)],), ({'plan': 'v3', 'code': 'x=1'}, ['coder'])),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Two dicts updated in a loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same, with `dict.get(key, 0)` for the version.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Two dicts updated in a loop.

```python
def apply_updates(updates):
    state, version, rejected = {}, {}, []
    for agent, key, value, seen in updates:
        current = version[key] if key in version else 0
        if seen == current:
            state[key] = value
            version[key] = current + 1
        else:
            rejected.append(agent)
    return (state, rejected)
```

**Cost:** O(u) time, O(k) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Same, with `dict.get(key, 0)` for the version.

```python
def apply_updates(updates):
    state, version, rejected = {}, {}, []
    for agent, key, value, seen in updates:
        if seen == version.get(key, 0):
            state[key], version[key] = value, seen + 1
        else:
            rejected.append(agent)
    return (state, rejected)
```

**Cost:** O(u) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Two dicts updated in a loop. That costs O(u) time and O(k) space."

"I can do better: Same, with `dict.get(key, 0)` for the version. That's O(u) time and O(k) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/multi-agent/shared-state-without-lost-updates)

<a id="q16"></a>
### Q16. Deliver Messages Between Agents

🟡 Interview standard · *Inboxes · queues*

Agents talk through inboxes. When an agent reads a message, `rules[agent]` lists who it forwards to. Starting with one message to `first`, process messages first-in-first-out and return the order in which agents read messages (stop after `limit` reads).

**Examples**

- `rules = {'user': ['planner'], 'planner': ['coder', 'tester'], 'coder': ['reviewer'], 'tester': [], 'reviewer': []}, first = 'user', limit = 10` → `['user', 'planner', 'coder', 'tester', 'reviewer']`
- `rules = {'a': ['b'], 'b': ['a']}, first = 'a', limit = 5` → `['a', 'b', 'a', 'b', 'a']`

In [ ]:
def deliver(rules, first, limit):
    """
    Agents talk through inboxes. When an agent reads a message, rules[agent]
    lists who it forwards to. Starting with one message to first, process
    messages first-in-first-out and return the order in which agents read
    messages (stop after limit reads).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(deliver, [
    (({'user': ['planner'], 'planner': ['coder', 'tester'], 'coder': ['reviewer'], 'tester': [], 'reviewer': []}, 'user', 10), ['user', 'planner', 'coder', 'tester', 'reviewer']),
    (({'a': ['b'], 'b': ['a']}, 'a', 5), ['a', 'b', 'a', 'b', 'a']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A list used as a queue (pop(0)).

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`collections.deque` for O(1) pops from the front.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** A list used as a queue (pop(0)).

```python
def deliver(rules, first, limit):
    queue, reads = [first], []
    while queue and len(reads) < limit:
        agent = queue.pop(0)
        reads.append(agent)
        for nxt in rules[agent]:
            queue.append(nxt)
    return reads
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `collections.deque` for O(1) pops from the front.

```python
def deliver(rules, first, limit):
    queue, reads = deque([first]), []
    while queue and len(reads) < limit:
        agent = queue.popleft()
        reads.append(agent)
        queue.extend(rules[agent])
    return reads
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A list used as a queue (pop(0)). That costs O(n²) time and O(n) space."

"I can do better: `collections.deque` for O(1) pops from the front. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/multi-agent/deliver-messages-between-agents)

<a id="q17"></a>
### Q17. What Do Extra Agents Cost?

🟡 Interview standard · *Token multiplication*

One agent solving a task reads `context` tokens once and writes `output`. An orchestrator with `n` workers: the orchestrator reads the context and writes a plan of `plan` tokens; each worker reads the context plus the plan and writes `output`; the orchestrator then reads all worker outputs and writes `output`. Return `(single total, multi total, ratio rounded to 1)`.

**Examples**

- `context = 20000, output = 1000, plan = 500, n = 4` → `(21000, 111500, 5.3)`
- `context = 2000, output = 800, plan = 300, n = 3` → `(2800, 14800, 5.3)`

In [ ]:
def multi_cost(context, output, plan, n):
    """
    One agent solving a task reads context tokens once and writes output. An
    orchestrator with n workers: the orchestrator reads the context and writes a
    plan of plan tokens; each worker reads the context plus the plan and writes
    output; the orchestrator then reads all worker outputs and writes output.
    Return (single total, multi total, ratio rounded to 1).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(multi_cost, [
    ((20000, 1000, 500, 4), (21000, 111500, 5.3)),
    ((2000, 800, 300, 3), (2800, 14800, 5.3)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Add up each agent's reads and writes step by step.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A closed-form expression.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Add up each agent's reads and writes step by step.

```python
def multi_cost(context, output, plan, n):
    single = context + output
    total = context + plan
    for _ in range(n):
        total += context + plan + output
    total += n * output + output
    return (single, total, round(total / single, 1))
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A closed-form expression.

```python
def multi_cost(context, output, plan, n):
    single = context + output
    multi = (context + plan) + n * (context + plan + output) + (n * output + output)
    return (single, multi, round(multi / single, 1))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Add up each agent's reads and writes step by step. That costs O(n) time and O(1) space."

"I can do better: A closed-form expression. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/multi-agent/what-do-extra-agents-cost)

<a id="q18"></a>
### Q18. Map–Reduce Over a Long Document

🟡 Interview standard · *How many model calls?*

Summarise `n_chunks` chunks: one call per chunk (map), then combine summaries `fan_in` at a time, level after level, until one remains (reduce). Return `(map calls, reduce calls, reduce levels)`.

**Examples**

- `n_chunks = 40, fan_in = 5` → `(40, 11, 3)`
- `n_chunks = 1, fan_in = 5` → `(1, 0, 0)`
- `n_chunks = 26, fan_in = 5` → `(26, 9, 3)`

In [ ]:
def map_reduce(n_chunks, fan_in):
    """
    Summarise n_chunks chunks: one call per chunk (map), then combine summaries
    fan_in at a time, level after level, until one remains (reduce). Return (map
    calls, reduce calls, reduce levels).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(map_reduce, [
    ((40, 5), (40, 11, 3)),
    ((1, 5), (1, 0, 0)),
    ((26, 5), (26, 9, 3)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Simulate each level: ⌈k / fan\_in⌉ calls until k = 1.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same loop with `-(-k // fan_in)` for ceiling division.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Simulate each level: ⌈k / fan\_in⌉ calls until k = 1.

```python
def map_reduce(n_chunks, fan_in):
    k, calls, levels = n_chunks, 0, 0
    while k > 1:
        groups = k // fan_in
        if k % fan_in:
            groups += 1
        calls += groups
        levels += 1
        k = groups
    return (n_chunks, calls, levels)
```

**Cost:** O(log n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Same loop with `-(-k // fan_in)` for ceiling division.

```python
def map_reduce(n_chunks, fan_in):
    k, calls, levels = n_chunks, 0, 0
    while k > 1:
        k = -(-k // fan_in)
        calls, levels = calls + k, levels + 1
    return (n_chunks, calls, levels)
```

**Cost:** O(log n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Simulate each level: ⌈k / fan\_in⌉ calls until k = 1. That costs O(log n) time and O(1) space."

"I can do better: Same loop with `-(-k // fan_in)` for ceiling division. That's O(log n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/multi-agent/map-reduce-over-a-long-document)

<a id="q19"></a>
### Q19. A Pool of Workers

🔴 Stretch · *Concurrency limits*

`n` workers take tasks in the given order; each task starts on whichever worker frees up first. Return the time all tasks finish (rate limits often cap concurrency like this).

**Examples**

- `durations = [5, 3, 8, 2, 4, 6], n = 2` → `17`
- `durations = [5, 3, 8, 2, 4, 6], n = 3` → `11`
- `durations = [4, 4], n = 5` → `4`

In [ ]:
def pool_finish(durations, n):
    """
    n workers take tasks in the given order; each task starts on whichever
    worker frees up first. Return the time all tasks finish (rate limits often
    cap concurrency like this).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(pool_finish, [
    (([5, 3, 8, 2, 4, 6], 2), 17),
    (([5, 3, 8, 2, 4, 6], 3), 11),
    (([4, 4], 5), 4),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Track each worker's free time in a list and pick the minimum each time.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A min-heap of free times.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Track each worker's free time in a list and pick the minimum each time.

```python
def pool_finish(durations, n):
    free = [0] * n
    for d in durations:
        i = free.index(min(free))
        free[i] += d
    return max(free)
```

**Cost:** O(t · n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A min-heap of free times.

```python
def pool_finish(durations, n):
    heap = [0] * n
    for d in durations:
        heapq.heappush(heap, heapq.heappop(heap) + d)
    return max(heap)
```

**Cost:** O(t log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Track each worker's free time in a list and pick the minimum each time. That costs O(t · n) time and O(n) space."

"I can do better: A min-heap of free times. That's O(t log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/multi-agent/a-pool-of-workers)

<a id="q20"></a>
### Q20. Run a Graph Workflow

🔴 Stretch · *State machines (LangGraph style)*

Run the workflow `GRAPH`: each node is a function of the state returning the next node or None. Visiting 'draft' adds 1 to `rounds` and sets `score` to the next value from `scores`. Return the list of nodes visited.

**Examples**

- `scores = [5, 9]` → `['start', 'draft', 'review', 'draft', 'review', 'publish']`
- `scores = [4, 5, 6, 7]` → `['start', 'draft', 'review', 'draft', 'review', 'draft', 'review', 'escalate']`

In [ ]:
GRAPH = {"start": lambda s: "draft",
         "draft": lambda s: "review",
         "review": lambda s: "publish" if s["score"] >= 8 else ("draft" if s["rounds"] < 3 else "escalate"),
         "publish": lambda s: None, "escalate": lambda s: None}

def run_graph(scores):
    """
    Run the workflow GRAPH: each node is a function of the state returning the
    next node or None. Visiting 'draft' adds 1 to rounds and sets score to the
    next value from scores. Return the list of nodes visited.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(run_graph, [
    (([5, 9],), ['start', 'draft', 'review', 'draft', 'review', 'publish']),
    (([4, 5, 6, 7],), ['start', 'draft', 'review', 'draft', 'review', 'draft', 'review', 'escalate']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A while loop with if-statements for the node's side effects.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A table of node actions plus the same loop — how graph frameworks separate state updates from routing.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** A while loop with if-statements for the node's side effects.

```python
def run_graph(scores):
    state = {"rounds": 0, "score": 0}
    it = iter(scores)
    node, path = "start", []
    while node is not None:
        path.append(node)
        if node == "draft":
            state["rounds"] += 1
            state["score"] = next(it)
        node = GRAPH[node](state)
    return path
```

**Cost:** O(steps) time, O(steps) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A table of node actions plus the same loop — how graph frameworks separate state updates from routing.

```python
def run_graph(scores):
    it = iter(scores)
    actions = {"draft": lambda s: s.update(rounds=s["rounds"] + 1, score=next(it))}
    state, node, path = {"rounds": 0, "score": 0}, "start", []
    while node:
        path.append(node)
        actions.get(node, lambda s: None)(state)
        node = GRAPH[node](state)
    return path
```

**Cost:** O(steps) time, O(steps) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A while loop with if-statements for the node's side effects. That costs O(steps) time and O(steps) space."

"I can do better: A table of node actions plus the same loop — how graph frameworks separate state updates from routing. That's O(steps) time and O(steps) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/multi-agent/run-a-graph-workflow)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. When should you NOT use multi-agent systems?</b></summary>

When a single call, a prompt chain or one agent with good tools does the job. Multiple agents multiply tokens and failure modes; use them only when sub-tasks are independent or need separate specialisation or context.

</details>

<details>
<summary><b>2. Router vs orchestrator?</b></summary>

A router classifies a request once and sends it to one handler. An orchestrator plans several sub-tasks at run time, delegates them to workers and combines the results.

</details>

<details>
<summary><b>3. How do you stop an agent looping forever?</b></summary>

Cap steps and tokens, detect repeated identical tool calls, and give the model a way to stop (or escalate to a human) when it's stuck.

</details>

<details>
<summary><b>4. What is a handoff?</b></summary>

Transferring the conversation from one agent to another more suitable one, along with the context it needs — ideally a summary plus recent messages.

</details>

<details>
<summary><b>5. How do you evaluate a multi-agent system?</b></summary>

End to end on realistic tasks: success rate, cost, latency and number of steps, plus traces to find which agent caused each failure.

</details>

<details>
<summary><b>6. Why does the critical path matter?</b></summary>

With parallel workers, total time is the longest chain of dependent tasks, not the sum of all tasks — so it tells you how much parallelism can actually speed things up.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: What makes something an agent
- [ ] I can explain: Patterns, from simple to autonomous
- [ ] I can explain: Planning, dependencies and parallelism
- [ ] I can explain: Handoffs, shared state and messages
- [ ] I can explain: Graph workflows (LangGraph and friends)
- [ ] I can explain: Cost, safety and when multi-agent is worth it

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) Deliver Messages Between Agents
- [ ] [Q17](#q17) What Do Extra Agents Cost?
- [ ] [Q18](#q18) Map–Reduce Over a Long Document
- [ ] [Q19](#q19) A Pool of Workers
- [ ] [Q20](#q20) Run a Graph Workflow

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.